# ĐỒ ÁN TỐT NGHIỆP: YoFaSoShop_VTON (Ứng dụng TMĐT Thời trang tích hợp Thử đồ ảo)
## NOTEBOOK THỬ NGHIỆM TUẦN 4 (21/09 - 27/09/2026)
**Sinh viên thực hiện:** Nguyễn Vũ Khanh - MSSV: 22115053122118 - Lớp: 22T1  
**Giảng viên hướng dẫn:** ThS. Phạm Thị Trà My - Khoa Công nghệ số, ĐH Sư phạm Kỹ thuật - ĐH Đà Nẵng  

---
### Mục tiêu công việc Tuần 4:
1. Chạy thử nghiệm bóc tách phông nền sản phẩm shop bằng **DeepFashion2 (YOLOv8-seg)**, lưu ảnh tách nền vào `results/segmented/` và trích xuất bảng màu chủ đạo (Color Palette).
2. Chạy chuỗi thử đồ ảo **IDM-VTON** trên ảnh người mẫu với GPU đám mây (Tesla T4 16GB VRAM).
3. Xuất và lưu trữ **2 định dạng kết quả** vào `results/tryon/`:
   - **Ảnh thử đồ đơn (Single Try-on):** Chỉ chứa người mẫu sau khi đã ướm trang phục mới (`tryon_result.png`).
   - **Bảng đối sánh 3 cột:** `[Người mẫu gốc | Sản phẩm bóc nền | Kết quả thử đồ AI]` (`week4_report_comparison.jpg`).
4. Tự động đóng gói và tải kết quả về máy tính cục bộ khi chạy trên Google Colab.


### Bước 1: Kiểm tra GPU Google Colab

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q ultralytics diffusers transformers accelerate onnxruntime opencv-python scikit-learn pillow matplotlib huggingface_hub gradio_client


### Bước 3: Tải trọng số DeepFashion2 YOLOv8-seg và khởi tạo thư mục dữ liệu


In [ ]:
import os, sys
from pathlib import Path

# 1. Tự động định vị Root directory chuẩn của Repo (Hỗ trợ cả Google Colab và Local VS Code)
curr_dir = os.getcwd()
if curr_dir.endswith(os.path.join("ml-training", "notebooks")) or curr_dir.endswith("notebooks"):
    # Đang mở từ thư mục notebooks cục bộ trên Windows/Linux -> Chuyển về gốc Repo
    project_root = Path(curr_dir).resolve().parents[1]
    os.chdir(project_root)
    print(f"[*] Đã chuyển thư mục làm việc về gốc Repo: {os.getcwd()}")
else:
    print(f"[*] Thư mục làm việc hiện tại: {os.getcwd()}")

# 2. Khởi tạo cấu trúc cây thư mục chuẩn cho cả Colab và Local
dirs_to_create = [
    "data/test_samples/garments/tops",
    "data/test_samples/garments/bottoms",
    "data/test_samples/garments/dresses",
    "data/test_samples/persons",
    "data/test_samples/results/segmented",
    "data/test_samples/results/tryon",
    "weights"
]
for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)
print("✅ Các thư mục lưu trữ dữ liệu và kết quả đã sẵn sàng!")

# 3. Tải trọng số YOLOv8s-seg DeepFashion2 nếu chưa có
weight_path = "weights/deepfashion2_yolov8s-seg.pt"
if not os.path.exists(weight_path) or os.path.getsize(weight_path) < 100000:
    print("Đang tải trọng số YOLOv8s-seg DeepFashion2 (~24MB)...")
    !wget -q -O weights/deepfashion2_yolov8s-seg.pt https://huggingface.co/Bingsu/adetailer/resolve/main/deepfashion2_yolov8s-seg.pt
    print("Tải hoàn tất trọng số!")
else:
    print("Trọng số YOLOv8s-seg đã sẵn sàng!")


### Bước 4: Upload và kiểm tra ảnh trang phục/người mẫu (chấp nhận tên file tùy ý)


In [ ]:
import io
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

# Bước 4 tự tạo thư mục để vẫn chạy được nếu runtime Colab vừa khởi động lại.
garment_dir = Path("data/test_samples/garments/uploaded")
person_dir = Path("data/test_samples/persons/uploaded")
garment_dir.mkdir(parents=True, exist_ok=True)
person_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Cell upload này cần chạy trên Google Colab.") from exc

print("[*] Đang nạp mô hình kiểm tra ảnh...")
fashion_detector = YOLO("weights/deepfashion2_yolov8s-seg.pt")
person_detector = YOLO("yolov8n-pose.pt")  # Dùng keypoint để tránh nhầm quần thành người.


def read_uploaded_image(filename, file_bytes):
    """Đọc và chuẩn hóa mọi định dạng ảnh Pillow hỗ trợ sang RGB."""
    try:
        with Image.open(io.BytesIO(file_bytes)) as probe:
            probe.verify()
        image = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return None, f"'{filename}' không phải file ảnh hợp lệ ({exc})"

    if image.width < 128 or image.height < 128:
        return None, f"'{filename}' quá nhỏ ({image.width}x{image.height}); cần ít nhất 128x128 px"
    return image, None


def best_person_detection(image):
    result = person_detector.predict(
        source=np.asarray(image), classes=[0], conf=0.25, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
        return 0.0, 0.0, 0

    confidences = result.boxes.conf.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()
    image_area = image.width * image.height
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    best_index = int(np.argmax(confidences))
    keypoint_confidences = result.keypoints.conf
    visible_keypoints = 0
    if keypoint_confidences is not None:
        visible_keypoints = int(
            (keypoint_confidences[best_index].cpu().numpy() >= 0.35).sum()
        )
    return (
        float(confidences[best_index]),
        float(areas[best_index] / image_area),
        visible_keypoints,
    )


def best_garment_detection(image):
    result = fashion_detector.predict(
        source=np.asarray(image), conf=0.25, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0

    confidences = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    best_index = int(np.argmax(confidences))
    class_id = int(classes[best_index])
    label = result.names[class_id] if isinstance(result.names, dict) else result.names[class_id]
    return str(label), float(confidences[best_index])


def validate_image_role(image, expected_role):
    if expected_role == "person":
        person_confidence, person_area_ratio, visible_keypoints = best_person_detection(image)
        if person_confidence < 0.35 or person_area_ratio < 0.08 or visible_keypoints < 5:
            return False, (
                "không nhận diện được người đủ rõ "
                f"(confidence={person_confidence:.2f}, vùng người={person_area_ratio:.0%}, "
                f"keypoints={visible_keypoints}/17)"
            ), None
        return True, (
            f"đã nhận diện person (confidence={person_confidence:.2f}, "
            f"keypoints={visible_keypoints}/17)"
        ), None

    garment_label, garment_confidence = best_garment_detection(image)
    if garment_label is None or garment_confidence < 0.25:
        return False, "không nhận diện được quần áo bằng DeepFashion2", None
    return True, (
        f"đã nhận diện garment '{garment_label}' "
        f"(confidence={garment_confidence:.2f}); Bước 5 sẽ tách riêng trang phục bằng mask"
    ), garment_label


def safe_stem(filename):
    stem = re.sub(r"[^0-9A-Za-z_-]+", "_", Path(filename).stem).strip("_")
    return stem[:80] or "uploaded_image"


def upload_validated_image(expected_role, destination_dir):
    role_name = "TRANG PHỤC" if expected_role == "garment" else "NGƯỜI MẪU"
    print(f"\n👉 Chọn ảnh {role_name}; tên file có thể tùy ý (PNG/JPG/JPEG/WEBP...):")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError(f"Chưa có file nào được chọn cho {role_name}.")

    rejected = []
    for filename, file_bytes in uploaded.items():
        image, image_error = read_uploaded_image(filename, file_bytes)
        if image_error:
            print(f"❌ {image_error}")
            rejected.append(filename)
            continue

        accepted, message, detected_label = validate_image_role(image, expected_role)
        if not accepted:
            print(f"❌ Từ chối '{filename}': {message}.")
            rejected.append(filename)
            continue

        destination = destination_dir / f"{expected_role}_{safe_stem(filename)}.png"
        image.save(destination, format="PNG")
        print(f"✅ Chấp nhận '{filename}': {message}.")
        print(f"   Đã chuẩn hóa và lưu tại: {destination}")
        return str(destination), detected_label

    raise ValueError(
        f"Không có ảnh {role_name} hợp lệ. Hãy chọn đúng loại ảnh rồi chạy lại Bước 4. "
        f"Các file đã từ chối: {', '.join(rejected)}"
    )


# Mỗi lần chạy cell, người dùng có thể chọn ảnh mới với bất kỳ tên file nào.
garment_path, garment_class_name = upload_validated_image("garment", garment_dir)
person_path, _ = upload_validated_image("person", person_dir)
garment_description = (garment_class_name or "clothing").replace("_", " ")

def garment_category_from_label(label):
    normalized = (label or "").lower()
    if "dress" in normalized:
        return "dress"
    if normalized in {"shorts", "trousers", "skirt"}:
        return "lower_body"
    return "upper_body"

garment_category = garment_category_from_label(garment_class_name)

print("\n--- THÔNG TIN DỮ LIỆU ĐÃ KIỂM TRA THÀNH CÔNG ---")
print(f"[+] Trang phục: {garment_path} ({os.path.getsize(garment_path)/1024:.1f} KB)")
print(f"[+] Loại đồ   : {garment_description}")
print(f"[+] Vùng try-on: {garment_category}")
print(f"[+] Người mẫu : {person_path} ({os.path.getsize(person_path)/1024:.1f} KB)")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(Image.open(garment_path))
ax[0].set_title(f"Trang phục: {Path(garment_path).name}", fontsize=12)
ax[0].axis("off")
ax[1].imshow(Image.open(person_path))
ax[1].set_title(f"Người mẫu: {Path(person_path).name}", fontsize=12)
ax[1].axis("off")
plt.tight_layout()
plt.show()


### Bước 5: Chạy thử nghiệm bóc tách nền sản phẩm (DeepFashion2 YOLOv8-seg & Color Palette)


In [ ]:
from ultralytics import YOLO
import cv2, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

model = YOLO("weights/deepfashion2_yolov8s-seg.pt")
print("Nạp YOLOv8s-seg thành công!")
print(f"Đang phân đoạn và tách nền: {garment_path}")

# Đảm bảo thư mục lưu trữ tồn tại
out_segmented_dir = "data/test_samples/results/segmented"
os.makedirs(out_segmented_dir, exist_ok=True)
out_nobg_path = os.path.join(out_segmented_dir, "garment_nobg.png")
out_mask_path = os.path.join(out_segmented_dir, "garment_mask.png")
out_isolated_path = os.path.join(out_segmented_dir, "garment_isolated_white.png")

# Dự đoán bằng YOLOv8-seg DeepFashion2
results = model.predict(source=garment_path, conf=0.25, verbose=False)
res = results[0]

orig_img = Image.open(garment_path).convert("RGB")
orig_np = np.array(orig_img)
h, w = orig_np.shape[:2]

if res.boxes is None or len(res.boxes) == 0:
    raise ValueError("DeepFashion2 không còn nhận diện được garment ở Bước 5. Hãy chạy lại Bước 4.")

box_confidences = res.boxes.conf.cpu().numpy()
box_classes = res.boxes.cls.cpu().numpy().astype(int)
detected_labels = [str(res.names[int(class_id)]) for class_id in box_classes]
matching_indices = [
    index for index, label in enumerate(detected_labels)
    if label == garment_class_name
]
candidate_indices = matching_indices or list(range(len(box_confidences)))
selected_index = max(candidate_indices, key=lambda index: float(box_confidences[index]))
selected_label = detected_labels[selected_index]
selected_confidence = float(box_confidences[selected_index])
garment_class_name = selected_label
garment_description = selected_label.replace("_", " ")
garment_category = garment_category_from_label(selected_label)
print(f"[*] Chọn mask '{selected_label}' có confidence={selected_confidence:.2f}")
print(f"[*] Định tuyến try-on theo vùng: {garment_category}")

if res.masks is not None and len(res.masks) > selected_index:
    mask_tensor = res.masks.data[selected_index].cpu().numpy()
    mask_resized = cv2.resize((mask_tensor * 255).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    binary_mask = (mask_resized > 127).astype(np.uint8)
    mask_y, mask_x = np.where(binary_mask > 0)
    if len(mask_x) == 0 or len(mask_y) == 0:
        raise ValueError("Mask DeepFashion2 rỗng; không thể tách trang phục.")

    # Crop sát garment để loại người/nền khỏi đầu vào IDM-VTON.
    padding = max(8, int(0.03 * max(w, h)))
    x1 = max(0, int(mask_x.min()) - padding)
    x2 = min(w, int(mask_x.max()) + padding + 1)
    y1 = max(0, int(mask_y.min()) - padding)
    y2 = min(h, int(mask_y.max()) + padding + 1)
    
    # Tạo ảnh RGBA trong suốt, làm mềm biên bằng Gaussian Blur
    rgba_full = np.zeros((h, w, 4), dtype=np.uint8)
    rgba_full[:, :, :3] = orig_np
    rgba_full[:, :, 3] = cv2.GaussianBlur(binary_mask * 255, (5, 5), 0)
    rgba = rgba_full[y1:y2, x1:x2]
    Image.fromarray(rgba).save(out_nobg_path)
    cv2.imwrite(out_mask_path, (binary_mask * 255).astype(np.uint8))

    alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
    isolated_rgb = (rgba[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
    Image.fromarray(isolated_rgb).save(out_isolated_path)
    garment_for_tryon_path = out_isolated_path
    print(f"✅ Đã tách riêng garment khỏi người/nền: {os.path.abspath(out_nobg_path)} ({os.path.getsize(out_nobg_path)/1024:.1f} KB)")
    print(f"✅ Đã tạo garment nền trắng cho IDM-VTON: {os.path.abspath(out_isolated_path)}")
    print(f"✅ Đã lưu mask phân đoạn nhị phân: {os.path.abspath(out_mask_path)} ({os.path.getsize(out_mask_path)/1024:.1f} KB)")
    
    # Trích xuất Color Palette (K-Means Clustering 3 màu)
    fg_pixels = orig_np[binary_mask > 0]
    if len(fg_pixels) > 50:
        sample_pixels = fg_pixels[::max(1, len(fg_pixels) // 1000)]
        kmeans = KMeans(n_clusters=min(3, len(sample_pixels)), random_state=42, n_init=5).fit(sample_pixels)
        colors = kmeans.cluster_centers_.astype(int)
        hex_codes = [f"#{c[0]:02X}{c[1]:02X}{c[2]:02X}" for c in colors]
        print(f"🎨 Bảng màu chủ đạo trích xuất: {hex_codes}")
    
    # Hiển thị trực quan 3 khung hình
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(orig_img); ax[0].set_title("1. Ảnh sản phẩm gốc")
    ax[1].imshow(binary_mask, cmap="gray"); ax[1].set_title("2. Mask phân đoạn DeepFashion2")
    ax[2].imshow(isolated_rgb); ax[2].set_title("3. Garment đã tách riêng")
    for a in ax: a.axis("off")
    plt.tight_layout()
    plt.show()
else:
    raise ValueError(
        f"DeepFashion2 nhận diện '{selected_label}' nhưng không tạo được segmentation mask. "
        "Không dùng threshold dự phòng vì có thể giữ lại người trong ảnh."
    )


### Bước 6: Thử đồ theo category (IDM-VTON cho áo, OOTDiffusion cho quần/váy/đầm)

> **Xác thực Hugging Face:** mở biểu tượng chìa khóa **Secrets** ở thanh trái Colab, tạo secret tên `HF_TOKEN`, dán token quyền `read`, rồi bật **Notebook access**. Không ghi token trực tiếp vào code.


In [ ]:
import time, os
from PIL import Image
import numpy as np

print("=" * 70)
print("KHỞI CHẠY CHUỖI THỬ ĐỒ ẢO THEO CATEGORY")
print(f"[*] Ảnh người mẫu      : {person_path}")
print(f"[*] Ảnh garment đã tách : {garment_for_tryon_path}")
print("=" * 70)

start_time = time.time()
tryon_dir = "data/test_samples/results/tryon"
os.makedirs(tryon_dir, exist_ok=True)
tryon_out_path = os.path.join(tryon_dir, "tryon_result.png")

output = None
tryon_backend = None
remote_error = None

def load_hf_token():
    token = os.environ.get("HF_TOKEN")
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or token
    except Exception:
        # Secret chưa tồn tại/chưa cấp quyền hoặc đang chạy ngoài Colab.
        pass
    return token


hf_token = load_hf_token()
if hf_token:
    print("[*] Đã tìm thấy HF_TOKEN trong Colab Secrets; request sẽ được xác thực.")
else:
    print("[!] Chưa có HF_TOKEN; request ZeroGPU sẽ dùng quota ẩn danh rất thấp.")

def extract_gradio_image_paths(value):
    paths = []
    if isinstance(value, str):
        return [value]
    if isinstance(value, dict):
        for key in ("path", "image"):
            if key in value:
                paths.extend(extract_gradio_image_paths(value[key]))
    if isinstance(value, (tuple, list)):
        for item in value:
            paths.extend(extract_gradio_image_paths(item))
    return list(dict.fromkeys(paths))


def detect_output_garment(image, expected_category):
    result = fashion_detector.predict(
        source=np.asarray(image), conf=0.20, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0

    labels = [str(result.names[int(class_id)]) for class_id in result.boxes.cls.cpu().numpy()]
    confidences = result.boxes.conf.cpu().numpy()
    matches = [
        (label, float(confidences[index]))
        for index, label in enumerate(labels)
        if garment_category_from_label(label) == expected_category
    ]
    return max(matches, key=lambda item: item[1]) if matches else (None, 0.0)


# Định tuyến model theo vùng trang phục; IDM-VTON Space chỉ tạo mask upper_body.
try:
    from gradio_client import Client, handle_file

    def create_space_client(space_id):
        if not hf_token:
            return Client(space_id)
        try:
            return Client(space_id, token=hf_token)
        except TypeError:
            # Tương thích gradio_client phiên bản cũ (trước khi hf_token đổi tên thành token).
            return Client(space_id, hf_token=hf_token)

    if garment_category == "upper_body":
        print("[*] Upper-body -> IDM-VTON Space")
        client = create_space_client("yisol/IDM-VTON")
        result = client.predict(
            dict={"background": handle_file(person_path), "layers": [], "composite": handle_file(person_path)},
            garm_img=handle_file(garment_for_tryon_path),
            garment_des=garment_description,
            is_checked=True,
            is_checked_crop=False,
            denoise_steps=30,
            seed=42,
            api_name="/tryon"
        )
        tryon_backend = "IDM-VTON (upper_body)"
    else:
        oot_category = {"lower_body": "Lower-body", "dress": "Dress"}[garment_category]
        print(f"[*] {garment_category} -> OOTDiffusion DressCode ({oot_category})")
        client = create_space_client("levihsu/OOTDiffusion")
        result = client.predict(
            vton_img=handle_file(person_path),
            garm_img=handle_file(garment_for_tryon_path),
            category=oot_category,
            n_samples=4,
            n_steps=30,
            image_scale=3,
            seed=-1,  # Mỗi lần chạy tạo candidate khác; tránh lặp lại sample shorts lỗi.
            api_name="/process_dc"
        )
        tryon_backend = f"OOTDiffusion ({oot_category})"

    result_img_paths = extract_gradio_image_paths(result)
    if not result_img_paths:
        raise RuntimeError(f"Không đọc được đường dẫn ảnh từ kết quả Gradio: {result!r}")

    if garment_category == "upper_body":
        output = Image.open(result_img_paths[0]).convert("RGB")
    else:
        candidate_reports = []
        for candidate_index, candidate_path in enumerate(result_img_paths, start=1):
            candidate_image = Image.open(candidate_path).convert("RGB")
            candidate_label, candidate_confidence = detect_output_garment(
                candidate_image, garment_category
            )
            candidate_reports.append(
                f"#{candidate_index}: {candidate_label or 'không nhận diện'} "
                f"({candidate_confidence:.2f})"
            )
            print(f"    Candidate {candidate_reports[-1]}")

            subtype_matches = (
                candidate_label == garment_class_name
                if garment_category == "lower_body"
                else candidate_label is not None
            )
            if subtype_matches:
                output = candidate_image
                print(f"[+] Chọn candidate #{candidate_index}: đúng subtype '{garment_class_name}'.")
                break

        if output is None:
            raise RuntimeError(
                f"OOTDiffusion không sinh đúng subtype '{garment_class_name}' sau "
                f"{len(result_img_paths)} candidate: {', '.join(candidate_reports)}"
            )
    print(f"[+] {tryon_backend} sinh ảnh thành công!")
except Exception as e:
    remote_error = e
    print(f"[-] Backend try-on cho '{garment_category}' thất bại: {e}")

# Không được chuyển quần/váy sang IDM-VTON upper-body vì sẽ thay nhầm vùng áo.
if output is None and garment_category != "upper_body":
    raise RuntimeError(
        f"Không thể chạy try-on {garment_category}. OOTDiffusion không tạo được đúng subtype, "
        f"đang lỗi hoặc đang bận: {remote_error}. "
        "Notebook đã dừng để tránh tạo kết quả sai vùng trang phục. Hãy chạy lại Bước 6 sau."
    )

# Chỉ giữ preview composite dự phòng cho upper-body khi IDM-VTON bận/mất mạng.
if output is None:
    print("[*] IDM-VTON không khả dụng; tạo preview composite upper-body (không phải diffusion try-on).")
    tryon_backend = "Upper-body composite preview"
    human_pil = Image.open(person_path).convert("RGB").resize((768, 1024))
    # Sử dụng ảnh đã bóc tách nền ở Bước 5
    nobg_path = out_nobg_path
    if os.path.exists(nobg_path):
        cloth_rgba = Image.open(nobg_path).convert("RGBA")
    else:
        cloth_rgba = Image.open(garment_path).convert("RGBA")
    
    cloth_size = (int(768 * 0.72), int(1024 * 0.45))
    paste_y = int(1024 * 0.32)
    cloth_resized = cloth_rgba.resize(cloth_size)
    human_canvas = human_pil.copy()
    paste_x = int((768 - cloth_resized.width) / 2)
    human_canvas.paste(cloth_resized, (paste_x, paste_y), mask=cloth_resized.split()[3])
    output = human_canvas

infer_time = time.time() - start_time

# LƯU RIÊNG ẢNH KẾT QUẢ THỬ ĐỒ (ẢNH 1/2 TRONG results/tryon/)
output.save(tryon_out_path, "PNG")
print(f"[+] Hoàn thành thử đồ AI sau: {infer_time:.2f} giây!")
print(f"✅ [1/2] Đã lưu RIÊNG ảnh thử đồ ảo (Single Try-on) vào:")
print(f"    -> {os.path.abspath(tryon_out_path)} ({os.path.getsize(tryon_out_path)/1024:.1f} KB)")

# Chuẩn bị các biến cho Bước 7
human_img = Image.open(person_path).convert("RGB")
cloth_img = Image.open(garment_for_tryon_path).convert("RGB")


### Bước 7: Xuất bảng ảnh đối sánh nghiệm thu Tuần 4


In [ ]:
import matplotlib.pyplot as plt

report_compare_path = "data/test_samples/results/tryon/week4_report_comparison.jpg"
os.makedirs(os.path.dirname(report_compare_path), exist_ok=True)

fig, ax = plt.subplots(1, 3, figsize=(18, 8))
ax[0].imshow(human_img); ax[0].set_title("1. Người mẫu gốc (YoFaSo User)", fontsize=14, fontweight="bold")
ax[1].imshow(cloth_img); ax[1].set_title("2. Trang phục shop (YoFaSo Garment)", fontsize=14, fontweight="bold")
ax[2].imshow(output); ax[2].set_title(f"3. Kết quả thử đồ AI ({tryon_backend} - {infer_time:.2f}s)", fontsize=14, fontweight="bold")
for a in ax: a.axis('off')
plt.tight_layout()
plt.savefig(report_compare_path, dpi=200, bbox_inches='tight')
plt.show()

print("="*75)
print("             TỔNG KẾT KẾT QUẢ THỬ NGHIỆM TUẦN 4 (YoFaSoShop_VTON)             ")
print("="*75)
print(f"- Trạng thái bóc tách nền: THÀNH CÔNG (DeepFashion2 YOLOv8-seg)")
print(f"- Backend try-on: {tryon_backend}")
print(f"- Category: {garment_category}; thời gian render: {infer_time:.2f} giây")
print(f"- Thư mục results/segmented/: Đã lưu garment mask, RGBA và garment nền trắng")
print(f"- Thư mục results/tryon/    : Đã lưu đủ 2 ảnh kết quả:")
print(f"    [1] Ảnh thử đồ đơn lẻ : {os.path.abspath(tryon_out_path)}")
print(f"    [2] Bảng đối sánh 3 cột: {os.path.abspath(report_compare_path)}")
print("="*75)


### Bước 8: Tải trọn bộ kết quả về máy tính cá nhân (Dành riêng cho Google Colab)
Nếu bạn chạy trên **Google Colab**, toàn bộ ảnh được lưu trữ trên máy chủ đám mây của Google. Chạy cell bên dưới để tự động nén toàn bộ thư mục `results/` và tải file `.zip` về thư mục **Downloads** trên máy tính của bạn!

In [ ]:
import shutil, os

results_dir = "data/test_samples/results"
zip_base_name = "yofaso_vton_week4_results"
zip_file = f"{zip_base_name}.zip"

# Nén toàn bộ thư mục kết quả
if os.path.exists(results_dir):
    shutil.make_archive(zip_base_name, 'zip', results_dir)
    print(f"📦 Đã nén thành công: {zip_file} ({os.path.getsize(zip_file)/1024:.1f} KB)")
    
    # Kích hoạt tải về tự động nếu chạy trên Colab
    try:
        from google.colab import files
        print("🚀 Đang gửi yêu cầu tải file zip về trình duyệt của bạn...")
        files.download(zip_file)
        print("👉 Bạn chỉ cần kiểm tra thư mục Downloads trên máy tính và giải nén dán vào data/test_samples/results/")
    except ImportError:
        print(f"💻 Bạn đang chạy cục bộ (Local). Toàn bộ ảnh đã được lưu trực tiếp tại:")
        print(f"   {os.path.abspath(results_dir)}")
else:
    print(f"⚠️ Chưa tìm thấy thư mục {results_dir} để đóng gói!")
